# Fase 3 — Notebook 05: Cálculo do IVS final (item 9 do pedido)

**Entrada:** `banco_de_dados/entrega_orientadora/Base_ELSI_70Municipios_Censo2022.db`, lida
pelo motor de especificações da Fase C (`scripts/ivs_especificacoes.py`), que reaproveita o
motor da Fase 2 (`scripts/fatorial_ampliada.py`) e a camada fatorial
(`src/ivs_censo/fatorial.py`). Este notebook **não copia nenhuma conta**: chama o motor,
confere que ele regrava os mesmos números já versionados em
`banco_de_dados/eda/ivs_especificacoes/` e organiza o resultado nas respostas que a
orientadora pediu (seções 2.1 e 2.3 de `docs/prompts/prompt_nb05_ivs_final_2026-09.md`).

Nenhuma escolha de método é feita aqui: onde há alternativa, as alternativas ficam lado a
lado (seção 9) e a decisão continua em aberto (seção 11).

## 1. O que a orientadora pediu, e onde está cada resposta

| Item | Pedido | Onde está a resposta |
|---:|---|---|
| 1 | Renda: original · sem extremo · mediana municipal | Seção 3 |
| 2 | Rotação: sem rotação · Varimax · promax | Seção 4 |
| 3 | Pesos na solução oblíqua (5 alternativas) | Seção 5 |
| 4 | O lixo entra no índice? | Seção 6 |
| 5 | O banheiro entra, e em qual versão? | Seção 7 |
| 6 | O que o índice acrescenta à renda | Seção 8 |
| 7 | As três inferências dos agentes, viradas verificação | Fase A — `docs/prompts/estado_nb05_2026-09.md` |
| 8 | Pendências pequenas (dicionário, deck EDA Central, reexecução eda) | Fase A |
| 9 | Este notebook, o relatório e os slides | Este documento inteiro |

A especificação de referência, pelos critérios da seção 2.2 do prompt, e as alternativas de
sensibilidade aparecem lado a lado na seção 9; a seção 10 mostra o IVS final da referência
por setor e por município; a seção 11 lista as perguntas que continuam da orientadora —
nenhuma delas é fechada aqui.

## 2. As especificações testadas, e quantas são elegíveis

### 2.1 Importa o motor da Fase C

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys                         # registrar scripts/ e src/ no caminho de importação
import io                          # capturar a saída de texto do motor da Fase C
import shutil                      # apagar a pasta temporária ao final da conferência
import hashlib                     # comparar bytes dos CSVs regravados com os do HEAD
import subprocess                  # ler o conteúdo de um arquivo no HEAD do git
import tempfile                    # pasta temporária para a regravação de conferência
import contextlib                  # redireciona o stdout do motor para a variável acima
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve().parents[1]     # notebooks/Fase3_EDA_ELSI -> raiz do repositório
assert (RAIZ / 'scripts' / 'ivs_especificacoes.py').exists(), f'raiz errada: {RAIZ}'
sys.path.insert(0, str(RAIZ / 'src'))
sys.path.insert(0, str(RAIZ / 'scripts'))
import ivs_especificacoes as ivs   # o motor da Fase C — nenhuma função é reescrita aqui

SAIDA = RAIZ / 'banco_de_dados' / 'eda' / 'ivs_especificacoes'
print('motor importado de', ivs.__file__)

motor importado de /Users/pedro/Documents/Iniciacao Cientifica/Projeto_IVS_Censo22/scripts/ivs_especificacoes.py


A célula 1 importa o motor da Fase C tal como ele está.

### 2.2 Roda o motor numa pasta temporária e confere reprodutibilidade

In [2]:
# ── Chama o motor da Fase C (pasta temporária, fora do repositório) ─────────────
tmp = Path(tempfile.mkdtemp(prefix='ivs_especificacoes_'))
ivs.SAIDA = tmp                    # nunca escreve na pasta versionada durante a conferência
ivs.FIG = tmp / 'figuras'

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    ivs.main()                     # a grade inteira: 18 bases, 324 especificações
saida_motor = buf.getvalue()
print(saida_motor)

trava C.5: {'peso_f1': 65.01, 'auc_indice': 0.8131, 'auc_renda': 0.8819}
original·lixo_com·banheiro_fora: n=87545, elegíveis=[] (10 s)
original·lixo_com·banheiro_v00495: n=75955, elegíveis=['promax', 'varimax'] (27 s)
original·lixo_com·banheiro_graduado: n=73878, elegíveis=[] (35 s)
original·lixo_sem·banheiro_fora: n=87545, elegíveis=['promax', 'varimax'] (52 s)
original·lixo_sem·banheiro_v00495: n=75955, elegíveis=['promax', 'varimax'] (69 s)
original·lixo_sem·banheiro_graduado: n=73878, elegíveis=['promax', 'varimax'] (84 s)
sem_extremo·lixo_com·banheiro_fora: n=87544, elegíveis=[] (92 s)
sem_extremo·lixo_com·banheiro_v00495: n=75954, elegíveis=['promax', 'varimax'] (4686 s)
sem_extremo·lixo_com·banheiro_graduado: n=73877, elegíveis=[] (4694 s)
sem_extremo·lixo_sem·banheiro_fora: n=87544, elegíveis=['promax', 'varimax'] (4711 s)
sem_extremo·lixo_sem·banheiro_v00495: n=75954, elegíveis=['promax', 'varimax'] (4727 s)
sem_extremo·lixo_sem·banheiro_graduado: n=73877, elegíveis=['promax',

A célula 2 é a íntegra do que `ivs.main()` imprime: a trava da seção C.5 do
prompt (conferida antes de gravar qualquer coisa — se não bater, o motor para sozinho e nada
é gravado), o progresso das 18 bases com as rotações elegíveis em cada uma, o total de
especificações e elegíveis, a razão de cada uma das quatro leituras de referência (seção 2.2)
e a conclusão do acréscimo à renda. Nenhum destes números é digitado — é a saída literal do
motor da Fase C, rodado de novo por esta sessão.

In [3]:
# ── Confere os CSVs regravados contra o HEAD do git ─────────────────────────────
arquivos = ['especificacoes.csv', 'pesos.csv', 'avaliacao.csv', 'acrescimo_renda.csv',
            'estabilidade.csv', 'referencia.csv']


def hash_arquivo(caminho: Path) -> str:
    return hashlib.sha256(caminho.read_bytes()).hexdigest()


def hash_no_head(nome: str) -> str:
    rel = f'banco_de_dados/eda/ivs_especificacoes/{nome}'
    r = subprocess.run(['git', 'show', f'HEAD:{rel}'], cwd=RAIZ, capture_output=True, check=True)
    return hashlib.sha256(r.stdout).hexdigest()


identico = {nome: hash_arquivo(tmp / nome) == hash_no_head(nome) for nome in arquivos}
for nome, ok in identico.items():
    print(f'{nome}: {"idêntico ao HEAD" if ok else "DIFERENTE do HEAD"}')

# ivs_referencia_por_setor.csv não é versionado (regra *_por_setor.csv); confere contra a
# execução anterior da própria Fase C, que continua no lugar de sempre (nunca foi tocada).
setor_novo, setor_velho = tmp / 'ivs_referencia_por_setor.csv', SAIDA / 'ivs_referencia_por_setor.csv'
setor_igual = hash_arquivo(setor_novo) == hash_arquivo(setor_velho) if setor_velho.exists() else None
print('ivs_referencia_por_setor.csv (fora do git) idêntico à execução anterior:', setor_igual)

ivs.SAIDA, ivs.FIG = SAIDA, SAIDA / 'figuras'   # devolve o motor ao estado normal
shutil.rmtree(tmp)                              # nada fica fora do repositório

assert all(identico.values()), 'CSV regravado difere do HEAD — travas_ok=false, parar e reportar'
assert setor_igual is not False, 'ivs_referencia_por_setor.csv mudou entre execuções'
print('\ntodos os CSVs regravados batem com a Fase C — o motor é reprodutível.')

especificacoes.csv: idêntico ao HEAD
pesos.csv: idêntico ao HEAD
avaliacao.csv: idêntico ao HEAD
acrescimo_renda.csv: idêntico ao HEAD
estabilidade.csv: idêntico ao HEAD
referencia.csv: idêntico ao HEAD
ivs_referencia_por_setor.csv (fora do git) idêntico à execução anterior: True

todos os CSVs regravados batem com a Fase C — o motor é reprodutível.


A célula 3 regrava a grade inteira numa pasta temporária e compara, byte a
byte, cada um dos seis CSVs versionados com a cópia que já está no HEAD do git — e o CSV por
setor (fora do git) contra a execução anterior da própria Fase C. Vieram todos idênticos: o
motor é determinístico (sem bootstrap, sementes fixas onde há alguma simulação — Horn usa
`semente=42`) e nenhuma das seções a seguir recalcula nada: todas leem os CSVs que a Fase C
já gravou em `banco_de_dados/eda/ivs_especificacoes/`.

### 2.3 Quantas especificações, e quantas elegíveis

In [4]:
esp = pd.read_csv(SAIDA / 'especificacoes.csv')
total = len(esp)
elegiveis = int(esp['elegivel'].sum())
por_norm = esp.groupby('normalizacao')['elegivel'].sum()
por_rotacao = esp.groupby('rotacao')['elegivel'].agg(elegiveis='sum', total='size')
print(f'{total} especificações; {elegiveis} elegíveis, por normalização:')
print(por_norm)
print('\npor rotação (elegíveis / total):')
print(por_rotacao)

324 especificações; 192 elegíveis, por normalização:
normalizacao
global       96
municipal    96
Name: elegivel, dtype: int64

por rotação (elegíveis / total):
             elegiveis  total
rotacao                      
promax             120    180
sem_rotacao          0     36
varimax             72    108


A célula 4 mostra o tamanho da grade (renda × lixo × banheiro × rotação ×
método de peso × normalização) e quantas especificações passam nos quatro critérios da seção
2.2 (KMO ≥ 0,70, MSA mínimo ≥ 0,50, comunalidade mínima ≥ 0,30, carga máxima mínima ≥ 0,40).
A quebra por rotação mostra o teste de "um fator ou dois": sem rotação (um fator só) elimina
a comunalidade mínima em toda a grade, como a Fase C já tinha registrado.

## 3. Renda: original · sem extremo · mediana municipal

In [5]:
por_renda = (esp[esp['elegivel']]
             .groupby('renda')['auc_mun_comum']
             .agg(especificacoes='size', auc_media='mean', auc_min='min', auc_max='max')
             .round(4))
por_renda

,especificacoes,auc_media,auc_min,auc_max
renda,,,,
mediana_mun,64,0.7968,0.7479,0.845
original,64,0.7964,0.7477,0.845
sem_extremo,64,0.7968,0.7479,0.845


A célula 5 compara, só entre as especificações elegíveis, a AUC contra FCU
dentro do município (amostra comum às três rendas) por versão de renda. As três alternativas
ficam próximas nesta medida — a escolha entre elas não é feita aqui (item 1 da seção 2.3): é
uma das perguntas que continuam da orientadora (seção 11).

## 4. Rotação: sem rotação · Varimax · promax

In [6]:
esp.groupby('rotacao')['elegivel'].agg(elegiveis='sum', total='size')

,elegiveis,total
rotacao,,
promax,120,180
sem_rotacao,0,36
varimax,72,108


A célula 6 conta, por rotação, quantas das 108 especificações (18 bases ×
6 combinações de método) são elegíveis. Sem rotação (um fator só) é sempre inelegível pela
comunalidade mínima; Varimax e promax elegem uma parte de cada base — a escolha entre as duas
também não é fechada aqui (item 2 da seção 2.3).

## 5. Pesos na solução oblíqua (promax): as cinco alternativas

In [7]:
promax = esp[esp['rotacao'] == 'promax']
(promax.groupby('metodo')
       .agg(elegiveis=('elegivel', 'sum'), total=('elegivel', 'size'),
            peso_f1_medio=('peso_f1', 'mean'), amplitude_media=('amp_peso_f1', 'mean'))
       .round(2))

,elegiveis,total,peso_f1_medio,amplitude_media
metodo,,,,
50_50,24,36,50.00,0.00
60_40,24,36,60.00,0.00
comunalidades,24,36,65.84,8.01
ss_estrutura,24,36,63.39,6.43
ss_padrao,24,36,67.60,6.18


A célula 7 mostra as cinco formas de repartir o peso entre as duas dimensões
na solução oblíqua (SS da matriz padrão, SS da estrutura, comunalidades da matriz oblíqua,
50/50 e 60/40 da literatura — item 3 da seção 2.3). Nos métodos de peso fixo (50/50, 60/40) a
amplitude ao deixar um município de fora é zero por construção, o que os favorece sempre no
critério (a) da seção 2.2 — registrado como pergunta na seção 11.

## 6. O lixo entra no índice?

In [8]:
esp.groupby('lixo').agg(elegiveis=('elegivel', 'sum'), total=('elegivel', 'size'),
                             comunalidade_min_media=('comunalidade_min', 'mean'))

,elegiveis,total,comunalidade_min_media
lixo,,,
com,48,162,0.288439
sem,144,162,0.355236


A célula 8 compara as bases com e sem o indicador de lixo inadequado. Com o
lixo, a comunalidade mínima da grade cai (a água inadequada costuma ficar abaixo de 0,30
nessas bases, como a Fase C já tinha registrado para o IVS-7 do NB04); sem o lixo, mais
especificações passam no critério de comunalidade. Se isso basta para o lixo sair do índice,
ou se a saída precisa de justificativa própria, é pergunta em aberto (seção 11).

## 7. O banheiro entra, e em qual versão?

In [9]:
(esp.groupby('banheiro')
    .agg(elegiveis=('elegivel', 'sum'), total=('elegivel', 'size'),
         faixas_indefinidas=('faixas_indefinidas', 'sum')))

,elegiveis,total,faixas_indefinidas
banheiro,,,
fora,48,108,0
graduado,48,108,108
v00495,96,108,0


A célula 9 compara as três versões do banheiro: fora do índice, V00495
("sem banheiro de uso exclusivo com chuveiro e vaso") e o graduado da Fase B
(`(1·V00236 + 2·V00237 + 3·V00238) / (3·V00001)`, só a partir da partição que a Fase B
provou fechar em 91.199 de 91.199 setores). A coluna de faixas indefinidas conta setores que
ficam sem faixa pela regra do IVS-BH por estarem no único município de sua faixa de setor
(DP indefinido) — mais frequente nas bases com o graduado, como a Fase C já tinha notado.

## 8. O que o índice acrescenta à renda (item 6 do pedido)

In [10]:
acr = pd.read_csv(SAIDA / 'acrescimo_renda.csv')
resumo = acr[['auc_indice_global', 'auc_renda_global', 'auc_indice_estratos',
              'auc_renda_estratos', 'queda_deviance', 'z_indice']].agg(['min', 'max', 'mean']).round(4)
print(resumo)
acr.loc[acr['referencia'], ['id', 'conclusao']]

      auc_indice_global  auc_renda_global  auc_indice_estratos  \
min              0.7980            0.8819               0.6805   
max              0.8663            0.8901               0.7364   
mean             0.8311            0.8878               0.7056   

      auc_renda_estratos  queda_deviance  z_indice  
min               0.5850         19.2803    7.8811  
max               0.5900       1457.5031   43.1010  
mean              0.5876        430.6855   21.3339  


,id,conclusao
62,original·lixo_sem·banheiro_graduado·promax·60_...,estratos de renda: índice 0.7062 × renda 0.590...


A célula 10 resume as 192 especificações elegíveis: a AUC do índice fica
sempre abaixo da renda invertida sozinha, global e na mediana municipal; dentro dos decis de
renda de cada município (colunas `_estratos`) o índice supera a renda em todas; e somar o
índice à renda na regressão logística sempre reduz a deviance (`queda_deviance` e `z_indice`
positivos em todas as linhas). A última linha mostra a conclusão específica da especificação
de referência (seção 9). **Limite:** FCU é critério do IBGE que já usa condições de moradia —
há circularidade parcial com as variáveis de saneamento do índice (item 6 da seção 2.3).

## 9. A referência proposta (seção 2.2) e as alternativas, lado a lado

In [11]:
ref = pd.read_csv(SAIDA / 'referencia.csv')
ref[['leitura', 'id', 'elegiveis', 'posto_a', 'posto_b', 'posto_c', 'posto_medio', 'porque']]

,leitura,id,elegiveis,posto_a,posto_b,posto_c,posto_medio,porque
0,principal,original·lixo_sem·banheiro_graduado·promax·60_...,96,24.5,43.0,48.5,38.666667,melhor posição média entre 96 elegíveis (norma...
1,amostra_propria,original·lixo_sem·banheiro_v00495·promax·60_40...,96,24.5,41.0,48.5,38.000000,melhor posição média entre 96 elegíveis (norma...
2,estabilidade_pesos_variaveis,sem_extremo·lixo_sem·banheiro_fora·promax·comu...,96,10.0,61.0,12.5,27.833333,melhor posição média entre 96 elegíveis (norma...
3,normalizacao_global,sem_extremo·lixo_sem·banheiro_fora·varimax·60_...,96,24.5,75.5,12.5,37.500000,melhor posição média entre 96 elegíveis (norma...


A célula 11 traz as quatro leituras possíveis do critério da seção 2.2 —
não só a leitura "principal" (como escrita: (a) amplitude do peso de F1; (b) AUC municipal na
amostra comum), mas três variantes de sensibilidade lado a lado, sem escolher entre elas:
`amostra_propria` troca a amostra comum pela amostra de cada especificação;
`estabilidade_pesos_variaveis` mede o critério (a) pela amplitude dos pesos das variáveis, não
do fator 1 (o que muda a proposta, porque nos pesos fixos a amplitude do peso de F1 é zero por
construção); `normalizacao_global` repete a leitura principal com a normalização global do
índice em vez da municipal. Isto é **proposta**, não decisão (seção 0.2 do prompt): a escolha
final entre as quatro, e entre cada uma delas e a leitura principal, continua com a
orientadora (seção 11).

## 10. O IVS final da especificação de referência

Sem geometria de setor censitário disponível no repositório, a distribuição de faixas fica em
tabela por município — a alternativa que a Fase D do prompt prevê para quando não há
geometria — em vez de mapa.

In [12]:
setor = pd.read_csv(SAIDA / 'ivs_referencia_por_setor.csv')
print('setores:', len(setor))
print('\ndistribuição do IVS de referência:')
print(setor['ivs_referencia'].describe().round(4))
print('\nfaixas pela regra do IVS-BH 2012:')
print(setor['faixa_ivs_bh'].value_counts())
print('\nfaixas por quartis municipais:')
print(setor['faixa_quartis'].value_counts())

setores:

 73878

distribuição do IVS de referência:
count    73878.0000
mean         0.3338
std          0.0822
min          0.0000
25%          0.2832
50%          0.3306
75%          0.3757
max          0.9120
Name: ivs_referencia, dtype: float64

faixas pela regra do IVS-BH 2012:
faixa_ivs_bh
medio            31052
baixo            21270
elevado          17622
muito_elevado     3933
Name: count, dtype: int64

faixas por quartis municipais:
faixa_quartis
muito_elevado    18493
medio            18472
elevado          18464
baixo            18449
Name: count, dtype: int64


A célula 12 descreve o IVS da especificação de referência (seção 9) para
todos os setores em que ela está definida, e conta quantos setores caem em cada uma das
quatro faixas pelas duas regras (IVS-BH 2012, seção 1 do prompt, e quartis municipais).

In [13]:
por_mun = (setor.groupby('NM_MUN')
           .agg(setores=('CD_SETOR', 'size'), ivs_medio=('ivs_referencia', 'mean'),
                pct_muito_elevado=('faixa_ivs_bh', lambda s: 100 * (s == 'muito_elevado').mean()))
           .sort_values('ivs_medio', ascending=False).round(4))
por_mun

,setores,ivs_medio,pct_muito_elevado
NM_MUN,,,
Rosário,15,0.5317,6.6667
Portel,17,0.5309,5.8824
Autazes,21,0.5300,4.7619
Placas,5,0.4903,0.0000
Japoatã,4,0.4767,0.0000
...,...,...,...
Curitiba,2206,0.2794,6.2103
Porto Alegre,1521,0.2699,6.1144
Canoas,376,0.2687,3.4574


A célula 13 é a tabela por município: os 70 municípios da entrega,
ordenados pelo IVS de referência médio, com a fração de setores na faixa "muito elevado". É a
tabela que substitui o mapa (sem geometria de setor no repositório).

## 11. As perguntas que continuam da orientadora

Nenhuma delas é fechada por este notebook (seção 0.2 do prompt: "não decidir pela
orientadora"). Cada uma aponta a seção com os números:

- **Renda** (seção 3): original, sem o extremo de Belo Horizonte, ou mediana municipal?
- **Rotação** (seção 4): sem rotação, Varimax ou promax?
- **Pesos na oblíqua** (seção 5): SS da matriz padrão, SS da estrutura, comunalidades, 50/50
  ou 60/40 da literatura?
- **Lixo** (seção 6): entra no índice ou não?
- **Banheiro** (seção 7): fora do índice, V00495 ou a versão graduada?
- **Normalização do índice**: dentro do município (referência) ou global (sensibilidade —
  seção 9, leitura `normalizacao_global`)?
- **Regra de faixas** (seção 10): IVS-BH 2012 (média ± desvio-padrão) ou quartis municipais?

A Fase C acrescentou variantes de leitura do próprio critério da seção 2.2 (seção 9) e um
ponto sobre o corte de comunalidade de 0,30 (o IVS-7 + banheiro graduado fica em 0,299, na
fronteira) — ambos registrados por extenso em `docs/prompts/estado_nb05_2026-09.md`, que esta
fase atualiza com a tabela completa dos itens 1–9.